In [44]:
# Run this if facing issue with transformers
# pip install torch torchvision torchaudio transformers

import pandas as pd
import numpy as np
import torch
import re
import random
from transformers import BertForSequenceClassification, BertTokenizer, AutoTokenizer, AutoModelForSequenceClassification, pipeline, Trainer, TrainingArguments
from datasets import Dataset
from scipy.stats import loguniform
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
#import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
# from sklearn.model_selection import ParameterGrid
# import tiktoken

In [2]:
train_df = pd.read_csv('Data Updated/train_df.csv')
val_df = pd.read_csv('Data Updated/val_df.csv')
test_df = pd.read_csv('Data Updated/test_df.csv')

In [3]:
# Apply preprocessing to datasets
def clean_text(text):
    if isinstance(text, str):
        # Convert to lowercase
        text = text.lower()
        # Remove special characters and punctuation
        text = re.sub(r"[^\w\s]", "", text)
        # Remove extra spaces
        text = re.sub(r"\s+", " ", text).strip()
    else:
        text = ""
    return text

def preprocess_dataset(df):
    df['title'] = df['title'].apply(clean_text)
    df['sentiment_label'] = df['sentiment_label'].astype(int)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle dataset
    return df

train_df = preprocess_dataset(train_df)
val_df = preprocess_dataset(val_df)
test_df = preprocess_dataset(test_df)

In [4]:
# # REMOVE LATER - for testing labelling only
# train_df = train_df.head(600)
# val_df = val_df.head(200)
# test_df = test_df.head(200)

# Prepare df for labelling results
test_df_results = test_df.copy()

In [5]:
# Check distribution of labels
print(train_df['sentiment_label'].value_counts())
print(val_df['sentiment_label'].value_counts())
print(test_df['sentiment_label'].value_counts())

sentiment_label
1    133131
0    128476
Name: count, dtype: int64
sentiment_label
1    32626
0    31459
Name: count, dtype: int64
sentiment_label
0    14673
1    13714
Name: count, dtype: int64


# FinBERT (Fine Tuned for Sentiment Analysis)
FinBERT is a pre-trained text analysis model, trained on financial data. It labels financial data as either "positive", "neutral" or "negative" by assigning a probability to each class, and return the class with the highest probability.
<br> For our use case, we labelled sentiments using only '1' or '0' for the excess 3 day aggregated returns. Hence, we will first fine tune finBERT to label using 2 classifications on our training set, then using random search to search for the best parameters that will return the model with the highest F1 by evaluating its performance on the valuation set.

## Hyperparameter Optimisation using Random Search (Key Evaluation Metric: f1)

In [8]:
# Select subset of train data for hyperparameter optimization
train_subset_df = train_df.sample(frac=0.1, random_state=42)

# If gpu available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = BertForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=2, ignore_mismatched_sizes=True).to(device)
model.config.problem_type = "single_label_classification" # To use finBERT for 2 labels

# Tokenize data
def preprocess_data(example):
    tokenized = tokenizer(example['title'], padding='max_length', max_length=128, truncation=True)
    tokenized['labels'] = example['sentiment_label']
    return tokenized

train_subset_tokenized_df = Dataset.from_pandas(train_subset_df).map(preprocess_data, batched=True)
train_subset_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_tokenized_df = Dataset.from_pandas(train_df).map(preprocess_data, batched=True)
train_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

val_tokenized_df = Dataset.from_pandas(val_df).map(preprocess_data, batched=True)
val_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

test_tokenized_df = Dataset.from_pandas(test_df).map(preprocess_data, batched=True)
test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Define random parameter space
random_param_space = {
    "learning_rate": loguniform(1e-6, 5e-5).rvs(50).tolist(),
    "batch_size": [16, 32],
    "num_epochs": [2, 3, 4],
    "threshold": np.random.uniform(0.1, 0.7, 50).tolist(),
    "weight_decay": [0.0, 0.01, 0.1]
}

num_random_samples = 50
random_param_combinations = [
    {
        "learning_rate": random.choice(random_param_space["learning_rate"]),
        "batch_size": random.choice(random_param_space["batch_size"]),
        "num_epochs": random.choice(random_param_space["num_epochs"]),
        "threshold": random.choice(random_param_space["threshold"]),
        "weight_decay": random.choice(random_param_space["weight_decay"]),
    }
    for _ in range(num_random_samples)
]

best_f1 = 0
best_params = None
best_threshold = None
best_model = None

# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred.predictions, eval_pred.label_ids
    predicted_probabilities = torch.softmax(torch.tensor(predictions), dim=1).numpy()

    # Apply threshold from params
    threshold = params["threshold"]
    predicted_labels = (predicted_probabilities[:, 1] >= threshold).astype(int)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_labels, average='binary')
    accuracy = accuracy_score(labels, predicted_labels)

    return {
        'eval_accuracy': accuracy,
        'eval_precision': precision,
        'eval_recall': recall,
        'eval_f1': f1
    }

# Perform random search
for params in random_param_combinations:
    print(f"Training with parameters: {params}")

    # Define training arguments
    training_args = TrainingArguments(
        output_dir='./train_subset_results',
        evaluation_strategy='epoch',  # Evaluates at the end of each epoch
        save_strategy='epoch',  # Saves the model at the end of each epoch
        logging_dir='./train_subset_logs',
        logging_steps=10,  # Log training loss every 10 steps
        logging_first_step=True,  # Log the first step to capture initial metrics
        per_device_train_batch_size=params['batch_size'],
        learning_rate=params['learning_rate'],
        num_train_epochs=params['num_epochs'],
        weight_decay=params['weight_decay'],
        load_best_model_at_end=True,
        metric_for_best_model='eval_f1',
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset_tokenized_df,
        eval_dataset=val_tokenized_df,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Train and evaluate
    trainer.train()
    eval_metrics = trainer.evaluate()
    print("Validation metrics:", eval_metrics)

    # Track best model
    if 'eval_f1' in eval_metrics and eval_metrics['eval_f1'] > best_f1:
        best_f1 = eval_metrics['eval_f1']
        best_metrics = eval_metrics
        best_params = params
        best_threshold = params['threshold']
        best_model = trainer.model
        trainer.save_model("./finbert_train_subset_model")
        tokenizer.save_pretrained("./finbert_train_subset_model")

print("Best F1 Score:", best_f1)
print("Best Eval Metrics:", best_metrics)
print("Best Parameters:", best_params)
print("Best Threshold:", best_threshold)

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/26161 [00:00<?, ? examples/s]

Map:   0%|          | 0/261607 [00:00<?, ? examples/s]

Map:   0%|          | 0/64085 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.132000,2.350606,0.504112,0.513514,0.493226,0.503166
2,0.200400,2.037835,0.503800,0.513864,0.469748,0.490817


Validation metrics: {'eval_accuracy': 0.504111726613092, 'eval_precision': 0.5135143759772792, 'eval_recall': 0.4932262612640226, 'eval_f1': 0.5031658927817645, 'eval_loss': 2.3506064414978027, 'eval_runtime': 63.9759, 'eval_samples_per_second': 1001.705, 'eval_steps_per_second': 125.219, 'epoch': 2.0}
Training with parameters: {'learning_rate': 2.595942550311263e-05, 'batch_size': 16, 'num_epochs': 2, 'threshold': 0.5564710291701385, 'weight_decay': 0.01}


C:\Users\Tylus\PycharmProjects\QF634_Research_Methods\.venv2\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Tylus\AppData\Local\Temp\ipykernel_14560\190666351.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.098500,3.535585,0.503768,0.513088,0.495648,0.504217
2,0.134600,3.184031,0.504221,0.513983,0.481089,0.496992


Validation metrics: {'eval_accuracy': 0.5037684325505188, 'eval_precision': 0.5130881746359108, 'eval_recall': 0.4956476429841231, 'eval_f1': 0.5042171398281964, 'eval_loss': 3.5355851650238037, 'eval_runtime': 64.3384, 'eval_samples_per_second': 996.061, 'eval_steps_per_second': 124.514, 'epoch': 2.0}
Best F1 Score: 0.6747321834801671
Best Eval Metrics: {'eval_accuracy': 0.5091519076226886, 'eval_precision': 0.5091289285602821, 'eval_recall': 1.0, 'eval_f1': 0.6747321834801671, 'eval_loss': 0.7248985171318054, 'eval_runtime': 64.9805, 'eval_samples_per_second': 986.219, 'eval_steps_per_second': 123.283, 'epoch': 4.0}
Best Parameters: {'learning_rate': 3.5067764992972196e-05, 'batch_size': 16, 'num_epochs': 4, 'threshold': 0.14442679104045422, 'weight_decay': 0.0}
Best Threshold: 0.14442679104045422


## Fine Tuning FinBERT using Optimised Hyperparameters on Training Set

In [25]:
# Training on full train set using best hyperparameters
params = best_params
def compute_metrics(eval_pred):
    predictions, labels = eval_pred.predictions, eval_pred.label_ids
    predicted_probabilities = torch.softmax(torch.tensor(predictions), dim=1).numpy()

    # Apply threshold from params
    threshold = params["threshold"]
    predicted_labels = (predicted_probabilities[:, 1] >= threshold).astype(int)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_labels, average='binary')
    accuracy = accuracy_score(labels, predicted_labels)

    return {
        'eval_accuracy': accuracy,
        'eval_precision': precision,
        'eval_recall': recall,
        'eval_f1': f1
    }

final_training_args = TrainingArguments(
    output_dir='./train_full_results',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./train_full_logs',
    logging_steps=10,
    per_device_train_batch_size=params['batch_size'],
    learning_rate=params['learning_rate'],
    num_train_epochs=params['num_epochs'],
    weight_decay=params['weight_decay'],
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    save_total_limit=1
)

final_trainer = Trainer(
    model=model,
    args=final_training_args,
    train_dataset=train_tokenized_df,  # Full train dataset
    eval_dataset=val_tokenized_df,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

final_trainer.train()
final_eval_metrics = final_trainer.evaluate()
print("Final Validation Metrics:", final_eval_metrics)
print(params)

# Save the fully trained model
final_trainer.save_model("./finbert_train_full_model")
tokenizer.save_pretrained("./finbert_train_full_model")


C:\Users\Tylus\PycharmProjects\QF634_Research_Methods\.venv2\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Tylus\AppData\Local\Temp\ipykernel_14560\4136196502.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.692400,0.693194,0.509105,0.509105,1.000000,0.674711
2,0.693300,0.693439,0.509105,0.509105,1.000000,0.674711
3,0.690500,0.693190,0.509105,0.509105,1.000000,0.674711
4,0.694400,0.692994,0.509105,0.509105,1.000000,0.674711


Final Validation Metrics: {'eval_accuracy': 0.5091050947959741, 'eval_precision': 0.5091050947959741, 'eval_recall': 1.0, 'eval_f1': 0.6747112531149507, 'eval_loss': 0.6931940317153931, 'eval_runtime': 64.544, 'eval_samples_per_second': 992.888, 'eval_steps_per_second': 124.117, 'epoch': 4.0}
{'learning_rate': 3.5067764992972196e-05, 'batch_size': 16, 'num_epochs': 4, 'threshold': 0.14442679104045422, 'weight_decay': 0.0}


('./finbert_train_full_model\\tokenizer_config.json',
 './finbert_train_full_model\\special_tokens_map.json',
 './finbert_train_full_model\\vocab.txt',
 './finbert_train_full_model\\added_tokens.json')

## Predicting on Test Set using Fine Tuned Model

In [42]:
# Use the final model to predict on test set
finbert_final_model = BertForSequenceClassification.from_pretrained('./finbert_train_full_model').to(torch.device('cuda'))
finbert_final_model.config.problem_type = "single_label_classification"

def predict_with_threshold(model, dataset, threshold):
    trainer = Trainer(model=model, tokenizer=tokenizer)
    predictions_output = trainer.predict(dataset)
    predictions = predictions_output.predictions
    labels = predictions_output.label_ids  # True labels from the dataset
    predicted_probabilities = torch.softmax(torch.tensor(predictions), dim=1).numpy()
    predicted_labels = (predicted_probabilities[:, 1] >= threshold).astype(int)

    return predicted_probabilities, predicted_labels, labels

# Use the best threshold for predictions
params = {'learning_rate': 1.5067764992972196e-05,
 'batch_size': 16,
 'num_epochs': 2,
 'threshold': 0.5,
 'weight_decay': 0.01}

threshold = params['threshold']
finbert_probabilities, finbert_predicted_labels, test_labels = predict_with_threshold(finbert_final_model, test_tokenized_df, threshold)

# Compute evaluation metrics on the test set
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, finbert_predicted_labels, average='binary')
accuracy = accuracy_score(test_labels, finbert_predicted_labels)

print(f"Test Accuracy: {accuracy}")
print(f"Test Precision: {precision}")
print(f"Test Recall: {recall}")
print(f"Test F1-Score: {f1}")

# Add predictions to the results dataframe
test_df_results['finbert_sentiment_label'] = finbert_predicted_labels
test_df_results

C:\Users\Tylus\AppData\Local\Temp\ipykernel_14560\2987283618.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, tokenizer=tokenizer)


Test Accuracy: 0.4831084651424948
Test Precision: 0.4831084651424948
Test Recall: 1.0
Test F1-Score: 0.6514809624474478


,Unnamed: 0,title,source,Date,sentiment_label,Ticker,date,finbert_sentiment_label
0,9704,tranche update on equity residential s equity ...,Capital IQ Buybacks Database,2024-11-04,0,EQR,2024-11-04,1
1,23443,tranche update on ralph lauren corporation s e...,Capital IQ Buybacks Database,2024-08-07,0,RL,2024-08-07,1
2,20426,northern trust corporation presents at barclay...,PR Newswire; Business Wire; GlobeNewswire; Can...,2024-09-10,0,NTRS,2024-09-10,1
3,23009,paypal holdings inc expected to report q3 2024...,Capital IQ Expected Earnings Database,2024-10-29,0,PYPL,2024-10-29,1
4,25698,truist financial corporation announces redempt...,PR Newswire,2024-11-01,0,TFC,2024-11-01,1
...,...,...,...,...,...,...,...,...
28382,22297,palantir technologies inc reports earnings res...,S&P Capital IQ Financials Database,2024-08-05,1,PLTR,2024-08-05,1
28383,5572,charter communications inc announces pricing f...,PR Newswire,2024-05-23,1,CHTR,2024-05-23,1
28384,890,ameren corporation shareholderanalyst call,SEC Filing,2024-05-09,0,AEE,2024-05-09,1
28385,16321,labcorp holdings inc 072 cash dividend nov262024,Financial Times,2024-11-26,0,LH,2024-11-26,1


In [13]:
# test_df_results.to_csv("test_df_results.csv", index=False)

C:\Users\Tylus\AppData\Local\Temp\ipykernel_12020\997101882.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, tokenizer=tokenizer)


In [ ]:
# # Compute ROC AUC
# roc_auc = roc_auc_score(test_labels, finbert_probabilities[:, 1])
# fpr, tpr, thresholds = roc_curve(test_labels, finbert_probabilities[:, 1])
#
# # Plot the ROC Curve
# plt.figure(figsize=(8, 6))
# plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.2f})", linewidth=2)
# plt.plot([0, 1], [0, 1], 'k--', label="Random Guess", linewidth=2)
# plt.title("ROC Curve", fontsize=16)
# plt.xlabel("False Positive Rate (FPR)", fontsize=14)
# plt.ylabel("True Positive Rate (TPR)", fontsize=14)
# plt.legend(loc="lower right", fontsize=12)
# plt.grid(True)
# plt.show()
#
# print(f"ROC AUC Score: {roc_auc}")